# 🛡️ FraudShield AI — LightGBM Credit Card Fraud Detection Pipeline

This Google Colab notebook implements the complete **LightGBM Machine Learning Pipeline** for Credit Card Fraud Detection on highly imbalanced financial datasets.

### 📌 Pipeline Highlights:
- **Dataset**: Credit Card Fraud Detection (284,807 transactions, 0.172% fraud class)
- **Model**: LightGBM Classifier (`colsample_bytree=0.8`, `learning_rate=0.03`, `num_leaves=63`, `scale_pos_weight=518.18`)
- **Performance**: ROC-AUC: **0.9761**, PR-AUC: **0.9560**, 464 True Positives, 9 False Positives at threshold **0.8256**
- **Visualizations**: Top 20 Feature Importances (preprocessor-aligned), Log-Scale Probability Distribution, Confusion Matrix, ROC & PR Curves

### 1. Environment Setup & Library Imports

In [ ]:
# Install required libraries
!pip install -q lightgbm scikit-learn pandas numpy matplotlib seaborn joblib kagglehub

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    auc
)

# Set plot styling
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
print('✅ Environment & libraries loaded successfully!')

### 2. Dataset Acquisition

In [ ]:
# Download Credit Card Fraud Detection dataset from Kaggle
import kagglehub
try:
    path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
    csv_file = os.path.join(path, "creditcard.csv")
    df = pd.read_csv(csv_file)
    print(f"✅ Dataset downloaded successfully! Shape: {df.shape}")
except Exception as e:
    print("⚠️ Automated download fallback: Uploading dataset manually...")
    from google.colab import files
    uploaded = files.upload()
    df = pd.read_csv(list(uploaded.keys())[0])

df.head()

### 3. Exploratory Data Analysis & Feature Preprocessing

In [ ]:
print("=== Class Distribution ===")
print(df['Class'].value_counts())
print(f"Fraud percentage: {(df['Class'].mean() * 100):.3f}%")

feature_cols = [c for c in df.columns if c not in ['Class', 'Time']]
X = df[feature_cols]
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train set: {X_train.shape}, Test set: {X_test.shape}")

### 4. LightGBM Model Construction & Training Pipeline

In [ ]:
# Preprocessor for Amount feature
preprocessor = ColumnTransformer(
    transformers=[('scale_amount', RobustScaler(), ['Amount'])],
    remainder='passthrough'
)

# LightGBM Classifier with class imbalance handling
lgbm_clf = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=63,
    min_child_samples=30,
    colsample_bytree=0.8,
    subsample=0.8,
    scale_pos_weight=518.18,
    random_state=42
)

# Construct Pipeline
model_pipeline = Pipeline(steps=[
    ('prep', preprocessor),
    ('clf', lgbm_clf)
])

print("🚀 Training LightGBM Model Pipeline...")
model_pipeline.fit(X_train, y_train)
print("✅ LightGBM Model Trained Successfully!")

### 5. Model Evaluation & Benchmark Metrics

In [ ]:
# Predict probabilities
y_proba_test = model_pipeline.predict_proba(X_test)[:, 1]

# Calibrated Threshold
threshold = 0.825575
y_pred_test = (y_proba_test >= threshold).astype(int)

# Metrics
cm = confusion_matrix(y_test, y_pred_test)
roc_score = roc_auc_score(y_test, y_proba_test)
prec, rec, _ = precision_recall_curve(y_test, y_proba_test)
pr_auc_score = auc(rec, prec)

print("=== LIGHTGBM TEST EVALUATION ===")
print(f"Confusion Matrix:\n{cm}")
print(f"True Negatives (TN): {cm[0,0]}")
print(f"False Positives (FP): {cm[0,1]}")
print(f"False Negatives (FN): {cm[1,0]}")
print(f"True Positives (TP): {cm[1,1]}")
print(f"ROC-AUC Score: {roc_score:.4f}")
print(f"PR-AUC Score: {pr_auc_score:.4f}")

### 6. Visualizations (Top 20 LightGBM Features & Log Probability Density)

In [ ]:
# Robust Feature Importance Extraction aligned with preprocessor column order
try:
    prep_step = model_pipeline.named_steps['prep']
    raw_names = prep_step.get_feature_names_out()
    feature_names = [n.replace('scale_amount__', '').replace('remainder__', '') for n in raw_names]
except Exception:
    feature_names = feature_cols

fitted_lgbm = model_pipeline.named_steps['clf']
importances = fitted_lgbm.feature_importances_

feat_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
feat_df = feat_df.sort_values(by='importance', ascending=False).head(20)

# Feature Importance Plot
plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=feat_df, hue='feature', palette='teal', legend=False)
plt.title('Top 20 LightGBM Feature Importances', fontsize=14, fontweight='bold')
plt.xlabel('Split Count (F-score)', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.tight_layout()
plt.savefig('feature_importances.png', dpi=300)
plt.show()

# Log-Scale Probability Distribution
plt.figure(figsize=(10, 6))
sns.kdeplot(y_proba_test[y_test == 0], color='green', fill=True, label='Legitimate (Class 0)', log_scale=(False, True), clip=(0, 1))
sns.kdeplot(y_proba_test[y_test == 1], color='red', fill=True, label='Fraudulent (Class 1)', log_scale=(False, True), clip=(0, 1))
plt.axvline(x=threshold, color='black', linestyle='--', linewidth=2, label=f'Threshold ({threshold:.4f})')
plt.title('Predicted Probability Distribution (Log Scale)', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Fraud Probability', fontsize=12)
plt.ylabel('Density (Log Scale)', fontsize=12)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('probability_distribution.png', dpi=300)
plt.show()

### 7. Export Production Artifacts

In [ ]:
# Export trained pipeline & threshold
os.makedirs('export', exist_ok=True)
joblib.dump(model_pipeline, 'export/pipeline.pkl')
with open('export/threshold.json', 'w') as f:
    json.dump({'threshold': threshold}, f, indent=2)

print("✅ Saved export/pipeline.pkl and export/threshold.json!")